---
title: The learning map
subtitle: Every lesson, and what it builds on
---

Each circle is a lesson, and each row is a level. Hover over a circle, tap it, or move to it with the Tab key. Violet marks every lesson it builds on, and cyan marks every lesson it unlocks. Click a circle to pin it, then open the lesson from the panel below the map. The chips filter by level, and the search box finds lessons by name or objective.

The lines show teaching order. They are not physical causes.

In [1]:
import sys, pathlib, subprocess
here = pathlib.Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "qcai").is_dir()), None)
if root is None:  # for example on Google Colab: fetch the course's small teaching package
    root = pathlib.Path("quantum-causal-ai")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/pozapas/quantum-causal-ai.git", str(root)], check=True)
    root = root.resolve()
sys.path.insert(0, str(root))

In [2]:
from qcai.widgets import learning_map
learning_map(base="../../")

## How many lessons each lab supports

Click a level to zoom in. The outer ring shows which lab goes with those lessons.

In [3]:
import collections, json
import plotly.graph_objects as go
from qcai.viz import show
from qcai.widgets import COLORS

levels = [json.loads((root / "_data" / "levels" / f"level{n}.json").read_text(encoding="utf-8")) for n in range(8)]
rows = collections.Counter((lv["level"], L["lab"]) for lv in levels for L in lv["lessons"])
ids, labels, parents, vals, colors = [], [], [], [], []
for k, lv in enumerate(levels):
    ids.append(f"L{k}"); labels.append(f"Level {k}"); parents.append(""); vals.append(0); colors.append(COLORS[k])
for (k, lab), c in sorted(rows.items()):
    ids.append(f"L{k}-{lab}"); labels.append(lab); parents.append(f"L{k}"); vals.append(c); colors.append(COLORS[k])
fig = go.Figure(go.Sunburst(ids=ids, labels=labels, parents=parents, values=vals, branchvalues="remainder",
                            marker=dict(colors=colors, line=dict(color="white", width=2)),
                            insidetextorientation="radial", hovertemplate="%{label}: %{value} lessons<extra></extra>"))
fig.update_layout(height=520, margin=dict(t=10, l=0, r=0, b=0), font=dict(family="Inter, Segoe UI, sans-serif"))
assert sum(rows.values()) == 80
show(fig)